<a href="https://colab.research.google.com/github/mireiialvarez/Exercise-2-/blob/main/Exercise_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##**Importing packages and corpus**



In [34]:
import numpy as np


In [4]:
#open, decode as utf-u and read the file
myfile = open("cat_news_2022.txt", 'r', encoding='utf‐8')
corpus = myfile.read()

In [5]:
print(len(corpus))

678766


## **Normalising the text before BPE implementation**

In [6]:
# Normalise the case, split the text into words and convert it into a character-level representation.
import re
def process(text):
  text = text.lower()
  # Remove numbers and punctuation (keep letters and spaces only)
  text = re.sub(r'[^a-z\s]', '', text)
  words= text.split ()
  token_list = []
  # Add an end of word token, so that BPE can respect word boundaries and avoid merging character pairs across words.
  for word in words:
        for char in word:
            token_list.append(char)
        token_list.append('</w>')
  # Extract all unique symbols from the corpus.
  char_unfiltered = list(set(token_list))
  # Keep only alphabetic characters and the end-of-word marker.
  vocab = [char for char in char_unfiltered if char.isalpha() or char == '</w>']
  # Return the character-level corpus and the filtered vocabulary.
  return token_list, vocab

In [7]:
# Check if the written code works properly by printing the first 30 characters.
token_list, vocab = process(corpus)
print (token_list,vocab [:30])

['p', 'r', 'o', 't', 'e', 'g', 'i', 'm', '</w>', 'l', 'o', 'c', 'u', 'p', 'a', 'c', 'i', '</w>', 'i', '</w>', 'l', 'l', 'u', 'i', 't', 'e', 'm', '</w>', 'c', 'o', 'n', 't', 'r', 'a', '</w>', 'l', 'e', 'm', 'p', 'i', 't', 'j', 'o', 'r', 'a', 'm', 'e', 'n', 't', '</w>', 'd', 'e', '</w>', 'l', 'e', 's', '</w>', 'n', 'o', 's', 't', 'r', 'e', 's', '</w>', 'c', 'o', 'n', 'd', 'i', 'c', 'i', 'o', 'n', 's', '</w>', 'd', 'e', '</w>', 'v', 'i', 'd', 'a', '</w>', 'h', 't', 't', 'p', 's', 't', '</w>', 'a', '</w>', 'a', 'n', 'g', 'l', 'a', 't', 'e', 'r', 'r', 'a', '</w>', 'e', 's', '</w>', 'v', 'a', '</w>', 'p', 'r', 'o', 'v', 'a', 'r', '</w>', 'd', 'e', '</w>', 'c', 'o', 'm', 'e', 'n', 'a', 'r', '</w>', 'm', 's', '</w>', 't', 'a', 'r', 'd', '</w>', 'l', 'i', 'n', 's', 't', 'i', 't', 'u', 't', '</w>', 'i', '</w>', 'e', 's', '</w>', 'v', 'a', '</w>', 'v', 'e', 'u', 'r', 'e', '</w>', 'q', 'u', 'e', '</w>', 'm', 'i', 'l', 'l', 'o', 'r', 'a', 'v', 'a', '</w>', 'e', 'l', '</w>', 'r', 'e', 'n', 'd', 'i',

##**Pair occurence**

In [8]:
# This function will pair the adjacent characters and count their frequency.
def get_pair_frequencies(token_list):
    # Create an empty dictionary to store pair frequencies.
    pair_freq = {}
    # Iterate over the character sequence and extract all adjacent symbol pairs by examining each position i and its subsequent position i+1.
    for i in range(len(token_list) - 1):
        pair = (token_list[i], token_list[i+1])
        # If a pair is encountered for the first time, initialise its count to 0, then increment its frequency.
        if pair not in pair_freq:
            pair_freq[pair] = 0
        pair_freq[pair] += 1
    # Return the dictionary containing the frequency of each adjacent pair.
    return pair_freq

In [9]:
# Testing pair_freq to check the code is working as expected.
pair_freq = get_pair_frequencies(token_list)
print(list(pair_freq.items())[:10])

[(('p', 'r'), 2932), (('r', 'o'), 2159), (('o', 't'), 1396), (('t', 'e'), 3878), (('e', 'g'), 1296), (('g', 'i'), 592), (('i', 'm'), 1495), (('m', '</w>'), 1651), (('</w>', 'l'), 8770), (('l', 'o'), 1112)]


##**Finding the most frequent pair**

In [10]:

def get_best_pair(pair_freq):
    # Select the adjacent symbol pair with the highest frequency by returning the dictionary key associated with the maximum value.
    return max(pair_freq, key=pair_freq.get)

In [11]:
# checking which is the most frequent pair from the corpus.
best_pair = get_best_pair(pair_freq)
print(best_pair)

('a', '</w>')


## **Merging**
It replaces every occurence of a specific adjacent pair with a merged symbol. For example, if the best pair is ('i','n'), it will be replaced by *in* everywhere in the corpus.

In [12]:
def merge_pair(token_list, pair):
    # Merge all occurrences of the selected adjacent symbol pair by replacing them with a new combined symbol.
    new_tokens = []
    i = 0 #index position where we start reading the corpus.

    while i < len(token_list):
        # If the current and next symbols match the pair, combine them into a single symbol.
        if i < len(token_list)-1 and token_list[i] == pair[0] and token_list[i+1] == pair[1]:
            new_tokens.append(pair[0] + pair[1])
            i += 2 #Skip both symbols since they have been merged.
        else:
            # Otherwise, copy the current symbol unchanged.
            new_tokens.append(token_list[i])
            i += 1 # Move to the next position.

    return new_tokens

In [13]:
# testing one merge
token_list = merge_pair(token_list, best_pair)
print(token_list[:50])

['p', 'r', 'o', 't', 'e', 'g', 'i', 'm', '</w>', 'l', 'o', 'c', 'u', 'p', 'a', 'c', 'i', '</w>', 'i', '</w>', 'l', 'l', 'u', 'i', 't', 'e', 'm', '</w>', 'c', 'o', 'n', 't', 'r', 'a</w>', 'l', 'e', 'm', 'p', 'i', 't', 'j', 'o', 'r', 'a', 'm', 'e', 'n', 't', '</w>', 'd']


##**Repeat (Loop)**

In [14]:
# These are the mergings I want to run. I will start with something small to see how it goes and then I will increment the number of merging to get a more precise result.
merge_settings = [100, 500, 1000]

In [15]:
# Create an empty dictionary to store results.
results = {}

In [32]:
# Process the corpus to obtain the character-level token sequence and the vocabulary.
token_list, vocab = process(corpus)
# Save a copy of the original tokens so the corpus can be reset before running each BPE experiment.
original_tokens = token_list.copy()

In [33]:
# These are the mergings I want to run. I will start with something small to see how it goes and then I will increment the number of merging to get a more precise result.
merge_settings = [100, 500, 1000]
# Create an empty dictionary to store results.
results = {}
for num_merges in merge_settings:

    print(f"\nRunning BPE with {num_merges} merges")
    token_list = original_tokens.copy()   # Store a copy of the starting corpus, before any experiments.
    merge_rules = []
    # Repeat the merge process for the specified number of iterations.
    for i in range(num_merges):
        #Compute frequencies of adjacent symbol pairs.
        # After each merge, the corpus changes, so we frequencies must be recomputed.
        pair_freq = get_pair_frequencies(token_list)
        #If there no more pairs to merge, stop early.
        if not pair_freq:
            break
        # Select the most frequent adjacent symbol pair.
        best_pair = get_best_pair(pair_freq)
        merge_rules.append(best_pair)
        # Replace all occurrences of that pair and update the corpus.
        token_list = merge_pair(token_list, best_pair)

    results[num_merges] = {
        "tokens": token_list,
        "merges": merge_rules
    }


Running BPE with 100 merges

Running BPE with 500 merges

Running BPE with 1000 merges


##**Results from the BPE**

In [35]:
# Print the first 50 learned subword units after 100 BPE merges.
for pair in results[100]["merges"][:50]:
    print(pair[0] + pair[1])

a</w>
s</w>
e</w>
en
t</w>
el
er
i</w>
es</w>
ar
an
qu
de</w>
al
es
el</w>
on
la</w>
un
in
or
que</w>
en</w>
tr
er</w>
at</w>
pr
am
ic
ac
ar</w>
re
it
om
al</w>
at
els</w>
ia</w>
di
per</w>
ol
est
ad
em
o</w>
ur
con
les</w>
ent</w>
b</w>


In [36]:
# Inspect merges 100–130 from the model trained with 500 merges.
for pair in results[500]["merges"][100:130]:
    print(pair[0] + pair[1])

als</w>
des
ada</w>
com</w>
il
est</w>
ter
or</w>
oc
ca
ats</w>
n</w>
pos
tre</w>
p</w>
itat</w>
ament</w>
dels</w>
ell
pres
tes</w>
por
ul
ver
tic
ib
ig
ut
aquest
a</w>l


In [37]:
# Inspect merges 500–530 from the model trained with 1000 merges.
for pair in results[1000]["merges"][500:530]:
    print(pair[0] + pair[1])

sor
primer</w>
ist
cal</w>
ar</w>l
aquest</w>di
president</w>
terr
aran</w>
val
ar</w>els</w>
lle
m</w>
cin
acab
int
temp
podr
istes</w>
at</w>la</w>
vin
difer
ni</w>
jor
primera</w>
cul
s</w>la</w>
acte</w>
i</w>els</w>
situ
